<a href="https://colab.research.google.com/github/ellenwterry/BioPhysics/blob/main/Normal_Process_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# EMERGENCE PRESENTATION: MOVIE - NORMAL IMMUNE SYSTEM PROCESS
# COMPLETE SINGLE-FILE MASTER SCRIPT
# ROBUST 3D POTENTIAL SURFACE, REAL DOMAIN EVALUATION & BALANCED KINETICS
# =====================================================================
!pip install pysindy networkx matplotlib scipy scikit-learn

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np
from scipy.integrate import solve_ivp

# Pitch black presentation theme & animation limit
plt.style.use('dark_background')
plt.rcParams['animation.embed_limit'] = 60.0

# ---------------------------------------------------------------------
# 1. ODE ENGINE: DECOUPLED BIOELECTRIC & BIOENERGETIC CASCADE
# ---------------------------------------------------------------------
t_full = np.linspace(0, 30, 160)

def physiological_ode(t, y):
    pax5, bcl6, irf4, blimp1, bach2, vmem, mito_fus = y
    dydt = np.zeros_like(y)

    # Extrinsic activation cue: T-cell Help / Signal 3 arriving at t >= 10s
    signal_3 = 1.0 / (1.0 + np.exp(-1.2 * (t - 10.0)))

    # Bioelectric Field: Autonomous electrophysiology uncoupled from BLIMP1
    v_target = -35.0 - 45.0 * signal_3
    dydt[5] = 0.85 * (v_target - vmem)

    # Voltage-Gated Ca2+ Influx & NFAT Permissive Drive (Levin Coupling)
    ca_influx = 3.5 / (1.0 + np.exp(0.12 * (vmem + 48.0)))
    nfat_drive = 2.4 * (ca_influx**3) / (1.2**3 + ca_influx**3)

    # Genetic Axis: Calibrated to contain IRF4 within biological bounds [0, 3.2]
    rep_shield = (2.2**2 / (2.2**2 + bcl6**2)) * (2.2**2 / (2.2**2 + pax5**2))

    # Balanced IRF4 toggle: strong enough to flip BLIMP1 and clear PAX5, but saturating at 3.0
    dydt[2] = (1.6 * signal_3 + nfat_drive) * rep_shield + 0.6 * (irf4**2 / (1.0**2 + irf4**2)) - 1.05 * irf4
    dydt[3] = 3.8 * (irf4**3 / (1.1**3 + irf4**3)) * (1.5**2 / (1.5**2 + bcl6**2)) * (1.0 / (1.0 + bach2**2)) - 1.0 * blimp1
    dydt[1] = 2.5 * (pax5**2 / (1.0 + pax5**2)) * (1.0 / (1.0 + blimp1**2)) * (1.0 / (1.0 + irf4**2)) - 0.9 * bcl6
    dydt[0] = 2.2 * (1.0 / (1.0 + blimp1**2)) - 0.95 * pax5
    dydt[4] = 0.8 * (bcl6**2 / (1.0 + bcl6**2)) * (1.0 / (1.0 + blimp1**2)) - 1.0 * bach2

    # Bioenergetic Axis: Mitochondrial Reticulation Grid driven by BLIMP1
    mito_target = 0.10 + 0.88 * (blimp1**2 / (1.0**2 + blimp1**2 + 1e-6))
    dydt[6] = 0.75 * (mito_target - mito_fus)

    return dydt

def run_simulation(init_state):
    sol = solve_ivp(
        physiological_ode,
        (t_full[0], t_full[-1]),
        init_state,
        t_eval=t_full,
        method='Radau'
    )
    return sol.y

# Primary baseline differentiation trajectory
test_init = np.array([2.2, 2.0, 0.05, 0.02, 0.20, -35.0, 0.12])
traj_primary = run_simulation(test_init)

# ---------------------------------------------------------------------
# 2. CONTINUOUS ATTRACTOR LANDSCAPE (ROBUST DUAL WELL & CANAL)
# ---------------------------------------------------------------------
traj_epi = (traj_primary[2, :] + traj_primary[3, :]) / (
    traj_primary[2, :] + traj_primary[3, :] + traj_primary[0, :] + traj_primary[1, :] + 1e-6
)
traj_vmem = traj_primary[5, :]

epi_grid = np.linspace(-0.06, 1.06, 70)
vmem_grid = np.linspace(-88.0, -28.0, 70)
EPI_mesh, VMEM_mesh = np.meshgrid(epi_grid, vmem_grid)

# Normalized coordinates: (0, 0) = GC basin, (1, 1) = Plasma basin
x_n = (EPI_mesh - 0.05) / 0.93
y_n = (VMEM_mesh - (-35.0)) / (-45.0)

# Potential Wells: deep minima for both terminal states
well_gc = 1.00 * np.exp(-((x_n**2) / 0.07 + (y_n**2) / 0.07))
well_plasma = 1.00 * np.exp(-(((x_n - 1.0)**2) / 0.07 + ((y_n - 1.0)**2) / 0.07))

# Robust curved canal using smooth quadratic path: safe across entire domain (no negative bases)
# Quadratic curve reflects bioelectric lead (y_n rises faster than x_n)
y_path = 1.4 * x_n - 0.4 * (x_n**2)
d_curved = y_n - y_path
canal_groove = 0.75 * np.exp(-(d_curved**2) / 0.06) * np.exp(-((x_n - 0.5)**2) / 0.45)

U_raw = 1.0 - (well_gc + well_plasma + canal_groove)
U_surface = (U_raw - np.min(U_raw)) / (np.max(U_raw) - np.min(U_raw))
U_surface = 0.05 + 0.95 * U_surface

# Evaluate trajectory altitude directly on the potential surface
traj_xn = (traj_epi - 0.05) / 0.93
traj_yn = (traj_vmem - (-35.0)) / (-45.0)

traj_ypath = 1.4 * traj_xn - 0.4 * (traj_xn**2)
traj_dcurved = traj_yn - traj_ypath

traj_well_gc = 1.00 * np.exp(-((traj_xn**2) / 0.07 + (traj_yn**2) / 0.07))
traj_well_plasma = 1.00 * np.exp(-(((traj_xn - 1.0)**2) / 0.07 + ((traj_yn - 1.0)**2) / 0.07))
traj_canal = 0.75 * np.exp(-(traj_dcurved**2) / 0.06) * np.exp(-((traj_xn - 0.5)**2) / 0.45)

traj_u_raw = 1.0 - (traj_well_gc + traj_well_plasma + traj_canal)
traj_u = (traj_u_raw - np.min(U_raw)) / (np.max(U_raw) - np.min(U_raw))
traj_u = 0.05 + 0.95 * traj_u

# ---------------------------------------------------------------------
# 3. PHYSICALLY ISOLATED ENTROPY DISSIPATION BUDGET (ΔS)
# ---------------------------------------------------------------------
traj_entropy = np.zeros(len(t_full))
for idx in range(len(t_full)):
    y_pt = traj_primary[:, idx]
    dy_pt = physiological_ode(t_full[idx], y_pt)

    # 1. Epigenetic transition dissipation (learning / prior erasure work)
    ds_epi = 0.50 * np.sum(np.abs(dy_pt[0:5]))

    # 2. Bioelectric holding maintenance (-80 mV ion pump floor)
    v_norm_pt = (y_pt[5] - (-35.0)) / (-45.0)
    ds_bio = 0.30 + 0.70 * max(0.0, v_norm_pt)

    # 3. Metabolic load: fusion work + active reticulated OXPHOS maintenance
    ds_met = 0.8 * (dy_pt[6]**2) + 1.2 * y_pt[6]

    # 4. Secretory Idling Work: UPR chaperone load & immunoglobulin pumping
    ds_secretory = 0.55 * y_pt[3] * y_pt[6]

    traj_entropy[idx] = ds_epi + ds_bio + ds_met + ds_secretory

# ---------------------------------------------------------------------
# 4. DASHBOARD VISUALIZATION & MASTER SLIDE LAYOUT
# ---------------------------------------------------------------------
fig = plt.figure(figsize=(19, 8.5), facecolor='#000000')

fig.suptitle("Movie: Normal Immune System Process", color='white', fontsize=18, y=0.96)
fig.subplots_adjust(top=0.86, wspace=0.28, bottom=0.10)

ax1 = fig.add_subplot(121, facecolor='#000000')
ax1_twin = ax1.twinx()
ax1_twin.set_facecolor('#000000')
ax2 = fig.add_subplot(122, projection='3d', facecolor='#000000')

# --- LEFT PANEL: MULTI-AXIS KINETIC TRACES ---
line_pax5, = ax1.plot([], [], color='#ff3366', lw=2.5, label='PAX5 (GC Preserver)')
line_bcl6, = ax1.plot([], [], color='#ff9900', lw=2.5, label='BCL6 (SHM Shield)')
line_irf4, = ax1.plot([], [], color='#00d4ff', lw=2.5, label='IRF4 (Inductive Switch)')
line_blimp1, = ax1.plot([], [], color='#00ff88', lw=2.5, label='BLIMP1 (Secretory Master)')
line_bach2, = ax1.plot([], [], color='#bd00ff', linestyle='-.', lw=2.0, label='Bach2 (GC Co-Factor)')
line_mfus, = ax1.plot([], [], color='#ffff00', linestyle='-', lw=2.0, label=r'Mito Fusion ($M_{\mathrm{fus}}$)')
line_entropy, = ax1.plot([], [], color='#ff00aa', linestyle=':', lw=2.6, label='Entropy Prod (ΔS)')
line_vmem, = ax1_twin.plot([], [], color='#70a5ff', linestyle='--', lw=2.2, label='Vmem (mV)')

ax1.axvline(10, color='#00ff88', linestyle='--', alpha=0.8, lw=1.5)
ax1.text(10.2, 5.0, 'Signal 3 Arrival (t=10s)', color='#00ff88', fontweight='bold', fontsize=8.5)

ax1.set_xlim(0, 30)
ax1.set_ylim(-0.2, 5.6)
ax1_twin.set_ylim(-92, -18)

ax1.set_xlabel('Time (s)', color='white', fontweight='bold', labelpad=8)
ax1.set_ylabel('Expression Level / Entropy Rate', color='white', fontweight='bold', labelpad=8)
ax1_twin.set_ylabel('Membrane Potential Vmem (mV)', color='#70a5ff', fontweight='bold', labelpad=12)

ax1.tick_params(colors='white')
ax1_twin.tick_params(axis='y', colors='#70a5ff')
ax1.grid(True, color='#222233', linestyle=':', alpha=0.6)

lines_all = [line_pax5, line_bcl6, line_irf4, line_blimp1, line_bach2, line_mfus, line_entropy, line_vmem]
labels_all = [l.get_label() for l in lines_all]
ax1.legend(lines_all, labels_all, loc='upper right', fontsize=8, facecolor='#0a0a0f', edgecolor='#333344')

# --- RIGHT PANEL: 3D ATTRACTOR LANDSCAPE ---
ax2.xaxis.set_pane_color((0.0, 0.0, 0.0, 1.0))
ax2.yaxis.set_pane_color((0.0, 0.0, 0.0, 1.0))
ax2.zaxis.set_pane_color((0.0, 0.0, 0.0, 1.0))
ax2.grid(color='#222233', linestyle='--', linewidth=0.5)

surf = ax2.plot_surface(
    EPI_mesh, VMEM_mesh, U_surface,
    cmap='viridis', edgecolor='#162032', lw=0.25, alpha=0.88,
    rstride=2, cstride=2, antialiased=True, shade=True
)

ax2.contour(
    EPI_mesh, VMEM_mesh, U_surface,
    zdir='z', offset=0.0, levels=14, cmap='viridis', alpha=0.50, linewidths=0.8
)

# Basin Anchors
ax2.scatter([traj_epi[0]], [traj_vmem[0]], [traj_u[0] + 0.02], color='#00ffff', s=180, edgecolors='white', lw=1.5, zorder=12)
ax2.text(traj_epi[0] + 0.04, traj_vmem[0], traj_u[0] + 0.08, 'GC Basin\n(-35mV)', color='#00ffff', fontsize=8, fontweight='bold', zorder=14)

ax2.scatter([traj_epi[-1]], [traj_vmem[-1]], [traj_u[-1] + 0.02], color='#ffcc00', s=180, edgecolors='white', lw=1.5, zorder=12)
ax2.text(traj_epi[-1] - 0.12, traj_vmem[-1] + 2.0, traj_u[-1] + 0.08, 'Plasma Basin\n(-80mV)', color='#ffcc00', fontsize=8, fontweight='bold', zorder=14)

trace_3d, = ax2.plot([], [], [], color='white', lw=3.5, zorder=15)
live_sphere, = ax2.plot([], [], [], color='#ff0055', marker='o', ms=10, markeredgecolor='white', markeredgewidth=1.5, zorder=20)

ax2.view_init(elev=28, azim=-128)
ax2.set_zlim(0.0, 1.05)

ax2.set_xlabel('Epigenetic Index (X)', color='#00d4ff', fontweight='bold', labelpad=10)
ax2.set_ylabel('Bioelectric Vmem (Y)', color='#ffcc00', fontweight='bold', labelpad=10)
ax2.set_zlabel('Quasi-Potential U(X,Y)', color='white', fontweight='bold', labelpad=8)
ax2.tick_params(colors='#888888')

# --- ANIMATION ENGINE ---
def update_dashboard(frame):
    t_curr = t_full[frame]

    # 1. Update Left Panel Traces
    line_pax5.set_data(t_full[:frame+1], traj_primary[0, :frame+1])
    line_bcl6.set_data(t_full[:frame+1], traj_primary[1, :frame+1])
    line_irf4.set_data(t_full[:frame+1], traj_primary[2, :frame+1])
    line_blimp1.set_data(t_full[:frame+1], traj_primary[3, :frame+1])
    line_bach2.set_data(t_full[:frame+1], traj_primary[4, :frame+1])
    line_mfus.set_data(t_full[:frame+1], traj_primary[6, :frame+1])
    line_entropy.set_data(t_full[:frame+1], traj_entropy[:frame+1])
    line_vmem.set_data(t_full[:frame+1], traj_primary[5, :frame+1])

    if t_curr <= 10.0:
        status_str = "Germinal Center Steady-State (SHM Active)"
    elif 10.0 < t_curr <= 18.0:
        status_str = "Physiological Differentiation: Bioelectric NFAT Transduction"
    else:
        status_str = "Terminal Plasma Cell Attractor (High Secretory Lock)"

    ax1.set_title(f"1. Multi-Axis Physical Kinetics\nTime: {t_curr:.1f}s | {status_str}", color='white', fontsize=9.5, fontweight='bold')

    # 2. Update Right Panel 3D Coordinate
    trace_3d.set_data(traj_epi[:frame+1], traj_vmem[:frame+1])
    trace_3d.set_3d_properties(traj_u[:frame+1] + 0.02)

    live_sphere.set_data([traj_epi[frame]], [traj_vmem[frame]])
    live_sphere.set_3d_properties([traj_u[frame] + 0.05])

    ax2.set_title(f"2. Distributed Attractor Landscape U(X, Y)\nState Vector Rolling Down Differentiation Valley", color='white', fontsize=9.5, fontweight='bold')

anim = FuncAnimation(fig, update_dashboard, frames=len(t_full), interval=50)
plt.close()

HTML(anim.to_jshtml())